# Delivery Time Prediction with TabPFN-3.5 - v2 (Colab T4/A100)
## Reference Evaluation with New Causal & Dynamic Features

This notebook runs the **v2 reference benchmark** of TabPFN-3.5 (`tabpfn 10.2.0`) on the Meituan delivery dataset using a Google Colab GPU (NVIDIA T4 / L4 / A100).

### Key Updates in v2:
1. **New Causal Dynamic & Courier Features (`FEATURES_COMPLETE_EXTRA`, 38 features)**:
   - Real-time district rolling ETA and delays (`district_recent_eta_30min`, `district_recent_delay_30min`).
   - Active restaurant queue load (`restaurant_active_orders`) and dynamic rolling prep times (`restaurant_recent_prep_min_60min`).
   - District courier supply-demand ratios (`supply_demand_ratio`, `supply_demand_ratio_order`).
   - Out-of-fold Bayesian smoothed courier encodings (`courier_target_enc`, `courier_speed_enc`).
2. **Dedicated Reference Scenario (`70_30` split, `complete_dispatch`)**:
   - Focuses 100% on the official reference scenario (`70_30` split, `manhattan` distance) over all 53,872 test orders.
3. **Optimized Large Context & Safe Batching**:
   - Uses `1 x N` large context (`TRAIN_SAMPLE_SIZE = 100,000` ou `75,000`) with `fit_mode='fit_with_cache'` and `output_type='median'`.
   - Controls test batch size (`batch_size=2048`) to eliminate cross-attention memory spikes on T4 (15 GB VRAM).
   - Robust OOM handling that breaks Python exception cycles and frees GPU cache before retry.
4. **Learning Curve & Context Scaling (10k -> 25k -> 50k -> 75k -> 100k)**:
   - Evaluates whether scaling beyond 50k continues to improve TabPFN accuracy and where it stands relative to LightGBM.
5. **Direct Baseline, Parity, and Blend Comparisons**:
   - Platform baseline (Meituan app estimate)
   - LightGBM on 377k (old 25 features vs new 38 features)
   - LightGBM on equal sample size (sample parity)
   - TabPFN-3.5 (median and mean outputs)
   - TabPFN + LightGBM weighted blend.


## Stage 0: Colab Setup, Authentication, and Project Bundle

Before running:
1. Select **Runtime -> Change runtime type -> T4 GPU** (or A100 / L4).
2. Add your `TABPFN_TOKEN` to Colab Secrets (key icon on the left sidebar).
3. If prompted, upload the updated `meituan_colab_bundle.zip` (generated via `python pack_colab_bundle.py`).
4. If you had a previous session running with an older `dataloader.py`, this cell automatically cleans the cache and unzips the new code.


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
# Reduz fragmentação da VRAM (precisa ser definido ANTES de importar torch)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
WORK_ROOT = Path('/content/meituan-project')
DRIVE_PROJECT_DIR = ''  # Optional: set to '/content/drive/MyDrive/...' if using Google Drive

# If dataloader was already imported in Python memory from an earlier run, clear it so the new one loads
for mod in ['dataloader', 'tabpfn_model']:
    if mod in sys.modules:
        del sys.modules[mod]

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabpfn==10.2.0', 'pyarrow', 'lightgbm'], check=True)

    if DRIVE_PROJECT_DIR and Path(DRIVE_PROJECT_DIR).exists():
        WORK_ROOT = Path(DRIVE_PROJECT_DIR)
    else:
        WORK_ROOT.mkdir(parents=True, exist_ok=True)
        local_zip = Path('/content/meituan_colab_bundle.zip')

        # Check if dataloader already exists and has the new features
        dataloader_file = WORK_ROOT / 'data' / 'dataloader.py'
        need_extract = True
        if dataloader_file.exists():
            content = dataloader_file.read_text(encoding='utf-8')
            if 'EXTRA_REDUCED_COLS' in content:
                print('dataloader.py atualizado já presente no disco.')
                need_extract = False

        if need_extract:
            if not local_zip.exists():
                from google.colab import files
                print('Por favor, selecione meituan_colab_bundle.zip atualizado para upload:')
                uploaded = files.upload()
                zip_names = [k for k in uploaded if k.endswith('.zip')]
                if not zip_names:
                    raise RuntimeError('Nenhum arquivo .zip foi enviado.')
                local_zip = Path('/content') / zip_names[0]

            print(f'Extraindo {local_zip.name} para {WORK_ROOT} (sobrescrevendo arquivos antigos)...')
            with zipfile.ZipFile(local_zip, 'r') as zf:
                zf.extractall(WORK_ROOT)

        # Sanity check: verify dataloader.py has the new features
        dl_text = (WORK_ROOT / 'data' / 'dataloader.py').read_text(encoding='utf-8')
        if 'EXTRA_REDUCED_COLS' not in dl_text:
            raise RuntimeError(
                "ATENÇÃO: O dataloader.py extraído é a versão ANTIGA (sem EXTRA_REDUCED_COLS).\n"
                "Solução: No seu computador, rode 'python pack_colab_bundle.py'.\n"
                "Em seguida, no Colab rode: '!rm -rf /content/meituan_colab_bundle.zip /content/meituan-project' e execute esta célula de novo enviando o novo zip."
            )

    nb_dir = WORK_ROOT / 'notebooks_analysis'
    nb_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(nb_dir)

    if not os.environ.get('TABPFN_TOKEN'):
        try:
            from google.colab import userdata
            token = userdata.get('TABPFN_TOKEN')
            if token:
                os.environ['TABPFN_TOKEN'] = token.strip()
        except Exception:
            pass

    if not os.environ.get('TABPFN_TOKEN'):
        raise RuntimeError('TABPFN_TOKEN not found. Add it to Colab Secrets.')
    print('Colab setup complete. Working dir:', Path.cwd())
else:
    print('Running outside Colab. Working dir:', Path.cwd())


## Stage 1: Hardware Detection, Hyperparameters & Memory Strategy

This section detects GPU memory, sets random seeds, imports modules from `dataloader.py` and `tabpfn_model.py`, and configures the execution parameters.

### Estratégia de memória para 100k linhas na T4 (15 GB):
- **Contexto fixo de 100.000 linhas** (`TRAIN_SAMPLE_SIZE = 100_000`) em qualquer GPU com >= 14 GB (T4, L4, A100).
- **`BATCH_SIZE = 2048`** no predict: a atenção cruzada teste x contexto fica 4x menor que com 8192.
- **`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`** (definido no Stage 0) reduz fragmentação da VRAM.
- **`run_with_oom_retry`**: em caso de OOM, libera a VRAM *fora* do bloco `except` (dentro dele o traceback ainda segura os tensores — foi isso que fez a curva do v1 dar OOM até com 5k), divide o lote por 2 e, se ainda assim não couber, troca `fit_mode` para `fit_preprocessors` (sem KV-cache: mais lento, bem menos memória).


In [ ]:
import gc
import json
import time
import zipfile
from dataclasses import replace

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tabpfn
import torch
from sklearn.ensemble import RandomForestRegressor

sys.path += ['../data', '../models']
from dataloader import (
    BASELINE,
    TARGET,
    FEATURES_COMPLETE,
    EXTRA_REDUCED_COLS,
    EXTRA_DISPATCH_COLS,
    FEATURES_COMPLETE_EXTRA,
    FEATURES_COMPLETE_EXTRA_CAT,
    CAT_COLS_BASE,
    CAT_COLS_COMPLETE_EXTRA,
    load_meituan,
)
from tabpfn_model import (
    TabPFNConfig,
    TabPFNModel,
    compute_metrics,
    sample_dataframe,
)

SEED = 42
CKPT_DIR = Path('../models/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
HAS_CUDA = torch.cuda.is_available()

if HAS_CUDA:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)')
    # Contexto de 100k linhas na T4 (15,6 GB reportados). GPUs menores caem para 50k.
    TRAIN_SAMPLE_SIZE = 100_000 if vram_gb >= 14.0 else 50_000
    N_ENSEMBLE = 1
    N_ESTIMATORS = 'auto'
    FIT_MODE = 'fit_with_cache'
    BATCH_SIZE = 2048  # 2048 garante que 100k rode sem OOM na T4 durante o predict
    TEST_EVAL_SIZE = None  # Avaliação completa (todas as 53.872 linhas de teste)
    LEARNING_CURVE_SIZES = [10_000, 25_000, 50_000, 75_000, 100_000]
    LC_EVAL_SIZE = 10_000  # Subconjunto estratificado de teste para curva rápida
else:
    print('Warning: CUDA GPU not detected; falling back to CPU profile (TabPFN-3.5 caps CPU at 5,000 rows).')
    gpu_name = 'CPU'
    vram_gb = 0.0
    TRAIN_SAMPLE_SIZE = 3000
    N_ENSEMBLE = 1
    N_ESTIMATORS = 8
    FIT_MODE = 'fit_with_cache'
    BATCH_SIZE = 2048
    TEST_EVAL_SIZE = 2000
    LEARNING_CURVE_SIZES = [1_000, 3_000]
    LC_EVAL_SIZE = 2000

OUTPUT_TYPE = 'median'             # Mediana minimiza MAE; 'mean' minimiza RMSE
CATEGORICAL_COLS = ('district_id',)  # Coluna categórica (district_id é identificador)
FEATURE_SET_NAME = 'FEATURES_COMPLETE_EXTRA'
FEAT_COLS = FEATURES_COMPLETE_EXTRA  # 38 features (25 originais + 7 dinâmicas + 6 entregador/despacho)

SAVE_JOBLIB_CHECKPOINTS = False


def free_gpu():
    """Force garbage collection and clear PyTorch CUDA allocator cache."""
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()


OOM_FALLBACK_FIT_MODES = ('fit_preprocessors',)  # sem KV-cache: mais lento, menos VRAM


def run_with_oom_retry(fn, cfg, min_batch=256, fallback_fit_modes=OOM_FALLBACK_FIT_MODES):
    """Call fn(cfg); on CUDA OOM halve batch_size, then fall back to cheaper fit_modes.

    GPU memory is released OUTSIDE the except block: inside it the traceback still
    references the frames (and CUDA tensors) of the failed call, so empty_cache()
    there frees nothing.
    """
    modes = [cfg.fit_mode] + [m for m in fallback_fit_modes if m != cfg.fit_mode]
    for mode in modes:
        bs = cfg.batch_size
        while bs >= min_batch:
            oom = False
            try:
                return fn(replace(cfg, batch_size=bs, fit_mode=mode))
            except torch.cuda.OutOfMemoryError:
                oom = True
            if oom:
                free_gpu()
                if HAS_CUDA:
                    print(f'[OOM] fit_mode={mode} batch_size={bs} '
                          f'(VRAM livre: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB)')
                bs //= 2
        print(f'[OOM] fit_mode={mode} não coube nem com batch_size={min_batch}.')
    raise torch.cuda.OutOfMemoryError(
        f'TabPFN não coube na GPU com sample_size={cfg.sample_size:,} em nenhum fit_mode {modes}.'
    )


plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'tabpfn={tabpfn.__version__} | context={N_ENSEMBLE}x{TRAIN_SAMPLE_SIZE:,} | '
      f'features={len(FEAT_COLS)} ({FEATURE_SET_NAME}) | output={OUTPUT_TYPE} | batch_size={BATCH_SIZE}')


## Stage 2: Data Loading (Reference Split: 70_30)

We load exclusively the official reference partition (`70_30` split, `distance='manhattan'`). This avoids loading unnecessary partitions, minimizing memory usage and startup delay.

The returned partitions contain 377,104 training orders and 53,872 validation and test orders, populated with the new causal and out-of-fold features.


In [ ]:
print('Loading reference split (70_30, distance=manhattan) with new features...')
t0 = time.time()
tr_70, val_70, te_70 = load_meituan(
    split='70_30',
    distance='manhattan',
    seed=SEED,
    verbose=True,
)
tr_70 = tr_70.copy()
val_70 = val_70.copy()
te_70 = te_70.copy()

# Add target quantile bins for stratified subsampling
tr_70['_y_bin'] = pd.qcut(tr_70[TARGET], q=20, labels=False, duplicates='drop')
val_70['_y_bin'] = pd.qcut(val_70[TARGET], q=20, labels=False, duplicates='drop')
te_70['_y_bin'] = pd.qcut(te_70[TARGET], q=20, labels=False, duplicates='drop')

te_eval, _, te_hash = sample_dataframe(
    te_70, TEST_EVAL_SIZE, stratify_col='_y_bin', seed=SEED
)
y_te = te_eval[TARGET].to_numpy()

print(f'\nData loaded in {time.time() - t0:.1f}s:')
print(f'  Train rows: {len(tr_70):,} | Validation rows: {len(val_70):,}')
print(f'  Test eval rows: {len(te_eval):,} (mean target = {y_te.mean():.2f} min, var = {y_te.var():.2f})')
print(f'  Baseline features (FEATURES_COMPLETE): {len(FEATURES_COMPLETE)} cols')
print(f'  Extra dynamic features (EXTRA_REDUCED_COLS): {len(EXTRA_REDUCED_COLS)} cols: {EXTRA_REDUCED_COLS}')
print(f'  Extra dispatch features (EXTRA_DISPATCH_COLS): {len(EXTRA_DISPATCH_COLS)} cols: {EXTRA_DISPATCH_COLS}')
print(f'  Total active feature set (FEATURES_COMPLETE_EXTRA): {len(FEATURES_COMPLETE_EXTRA)} cols')


## Stage 3: Platform Baseline Reference

We compute the error metrics of Meituan's own delivery estimate (`meituan_baseline_min`) on the test partition as our primary reference baseline.


In [ ]:
m_base = compute_metrics(y_te, te_eval[BASELINE])
baseline_df = pd.DataFrame([{
    'split': '70_30',
    'stage': 'platform_baseline',
    'model': 'Meituan App Estimate',
    'train_rows': 0,
    'features': 0,
    'test_MAE': round(m_base['MAE'], 3),
    'test_RMSE': round(m_base['RMSE'], 3),
    'test_R2': round(m_base['R2'], 4),
    'runtime_s': 0.0,
}])
display(baseline_df)


## Stage 4: Feature Impact Benchmark with Full LightGBM (377k Rows)

Before running TabPFN, we train LightGBM on all 377,104 training rows under two conditions:
1. **Original Baseline Features (`FEATURES_COMPLETE`, 25 features)**: Reproduces the original LightGBM reference (MAE ~5.313).
2. **New Extra Features (`FEATURES_COMPLETE_EXTRA`, 38 features)**: Quantifies the exact predictive gain contributed by the new features.


In [ ]:
# 1. LightGBM (full 377k, original 25 features)
print('Training LightGBM (full 377k) with original 25 features (FEATURES_COMPLETE)...')
t0 = time.time()
lgbm_orig = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.05, num_leaves=31, random_state=SEED, n_jobs=-1, verbose=-1
)
lgbm_orig.fit(
    tr_70[FEATURES_COMPLETE], tr_70[TARGET],
    eval_set=[(val_70[FEATURES_COMPLETE], val_70[TARGET])],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
m_lgbm_orig = compute_metrics(y_te, lgbm_orig.predict(te_eval[FEATURES_COMPLETE]))
t_lgbm_orig = round(time.time() - t0, 1)

# 2. LightGBM (full 377k, new 38 features)
print('Training LightGBM (full 377k) with new 38 features (FEATURES_COMPLETE_EXTRA)...')
t0 = time.time()
lgbm_extra = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.05, num_leaves=31, random_state=SEED, n_jobs=-1, verbose=-1
)
lgbm_extra.fit(
    tr_70[FEATURES_COMPLETE_EXTRA], tr_70[TARGET],
    eval_set=[(val_70[FEATURES_COMPLETE_EXTRA], val_70[TARGET])],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
lgbm_extra_te = lgbm_extra.predict(te_eval[FEATURES_COMPLETE_EXTRA])
lgbm_extra_val = lgbm_extra.predict(val_70[FEATURES_COMPLETE_EXTRA])
m_lgbm_extra = compute_metrics(y_te, lgbm_extra_te)
t_lgbm_extra = round(time.time() - t0, 1)

lgbm_comp_df = pd.DataFrame([
    {
        'model': 'LightGBM (377k, original features)',
        'features': len(FEATURES_COMPLETE),
        'test_MAE': round(m_lgbm_orig['MAE'], 3),
        'test_RMSE': round(m_lgbm_orig['RMSE'], 3),
        'test_R2': round(m_lgbm_orig['R2'], 4),
        'runtime_s': t_lgbm_orig,
    },
    {
        'model': 'LightGBM (377k, new features)',
        'features': len(FEATURES_COMPLETE_EXTRA),
        'test_MAE': round(m_lgbm_extra['MAE'], 3),
        'test_RMSE': round(m_lgbm_extra['RMSE'], 3),
        'test_R2': round(m_lgbm_extra['R2'], 4),
        'runtime_s': t_lgbm_extra,
    },
])
display(lgbm_comp_df)
print(f"\n-> Feature impact: MAE improved by {m_lgbm_orig['MAE'] - m_lgbm_extra['MAE']:+.3f} min")


## Stage 5: TabPFN-3.5 Reference Evaluation (New Features & Large Context)

We evaluate TabPFN-3.5 on the reference scenario (`70_30` split, `complete_dispatch`) with `FEATURES_COMPLETE_EXTRA` (38 features) using a single large context (`TRAIN_SAMPLE_SIZE = 100,000` or `75,000`).

- **Inference with Cache (`fit_mode='fit_with_cache'`)**: Encodes the training context once and caches it in VRAM.
- **Median vs Mean Output**: TabPFN outputs the full predicted distribution; `median` directly optimizes MAE, while `mean` optimizes RMSE. We extract both to compare.
- **Audit Manifest**: Generates a cryptographically verified SHA-256 audit manifest and saves prediction Parquet files.


In [ ]:
BASE_CONFIG = TabPFNConfig(
    model_version='v3.5',
    n_ensemble=N_ENSEMBLE,
    sample_size=TRAIN_SAMPLE_SIZE,
    sampling_strategy='stratified',
    n_estimators=N_ESTIMATORS,
    fit_mode=FIT_MODE,
    n_preprocessing_jobs=1,
    batch_size=BATCH_SIZE,
    aggregation='mean',
    output_type=OUTPUT_TYPE,
    categorical_cols=CATEGORICAL_COLS,
    ignore_pretraining_limits=True,
    seed=SEED,
    device='auto',
    verbose=True,
)
TABPFN_LABEL = f'TabPFN-{BASE_CONFIG.model_version} (1x{TRAIN_SAMPLE_SIZE:,}, {OUTPUT_TYPE})'

tag = f'colab_v2_70_30_ref_{TRAIN_SAMPLE_SIZE}'
print(f'\n--- Running TabPFN-3.5 Reference Benchmark ({len(FEAT_COLS)} features, n={TRAIN_SAMPLE_SIZE:,}) ---')

def _fit_and_predict(cfg):
    mdl = TabPFNModel(cfg).fit(
        tr_70,
        tr_70[TARGET],
        num_cols=FEAT_COLS,
        stratify_col='_y_bin',
        log_path=CKPT_DIR / f'{tag}_log.csv',
    )
    preds = mdl.predict(te_eval, output_type=OUTPUT_TYPE)
    return mdl, preds

if HAS_CUDA:
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
model, tab_preds_median = run_with_oom_retry(_fit_and_predict, BASE_CONFIG)
model.record_eval_target(te_eval[TARGET])
elapsed = round(time.time() - t0, 1)
USED_FIT_MODE, USED_BATCH_SIZE = model.config.fit_mode, model.config.batch_size
if HAS_CUDA:
    print(f'Pico de VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB | '
          f'fit_mode={USED_FIT_MODE} | batch_size={USED_BATCH_SIZE}')

# Extract mean predictions and validation predictions (reusing fitted model cache)
tab_preds_mean = model.predict(te_eval, output_type='mean')
tab_val_preds = model.predict(val_70)

m_tab_median = compute_metrics(y_te, tab_preds_median)
m_tab_mean = compute_metrics(y_te, tab_preds_mean)

manifest = model.save_audit_manifest(
    CKPT_DIR / f'{tag}_audit.json',
    extra={
        'split': '70_30',
        'stage': 'complete_dispatch_v2',
        'feature_set': FEATURE_SET_NAME,
        'features_count': len(FEAT_COLS),
        'sample_size': TRAIN_SAMPLE_SIZE,
        'distance': 'manhattan',
        'test_eval_sha256': te_hash,
        'test_metrics_median': m_tab_median,
        'test_metrics_mean': m_tab_mean,
        'runtime_s': elapsed,
        'fit_mode_used': USED_FIT_MODE,
        'batch_size_used': USED_BATCH_SIZE,
    },
)

# Save predictions
pd.DataFrame({
    'y_true': y_te,
    'y_pred_median': tab_preds_median,
    'y_pred_mean': tab_preds_mean,
}).to_parquet(CKPT_DIR / f'{tag}_predictions.parquet', index=False)

tab_res_df = pd.DataFrame([
    {
        'model': f'TabPFN-v3.5 (1x{TRAIN_SAMPLE_SIZE:,}, mean)',
        'test_MAE': round(m_tab_mean['MAE'], 3),
        'test_RMSE': round(m_tab_mean['RMSE'], 3),
        'test_R2': round(m_tab_mean['R2'], 4),
        'runtime_s': elapsed,
    },
    {
        'model': f'TabPFN-v3.5 (1x{TRAIN_SAMPLE_SIZE:,}, median)',
        'test_MAE': round(m_tab_median['MAE'], 3),
        'test_RMSE': round(m_tab_median['RMSE'], 3),
        'test_R2': round(m_tab_median['R2'], 4),
        'runtime_s': elapsed,
    },
])
display(tab_res_df)

del model
free_gpu()


## Stage 6: Sample-Parity Comparison & TabPFN + LightGBM Blend

In this stage we evaluate:
1. **Sample Parity**: LightGBM trained on the exact same `TRAIN_SAMPLE_SIZE` rows with the new features.
2. **Full-Data Baseline**: LightGBM trained on all 377,104 rows.
3. **TabPFN + LightGBM Blend**: An optimal convex combination $w \cdot y_{\text{TabPFN}} + (1 - w) \cdot y_{\text{LightGBM}}$ where $w$ is chosen to minimize validation MAE.


In [ ]:
# 1. LightGBM on equal sample size (sample parity)
tr_sub, _, _ = sample_dataframe(tr_70, TRAIN_SAMPLE_SIZE, stratify_col='_y_bin', seed=SEED)
t0 = time.time()
lgbm_sub = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.05, num_leaves=31, random_state=SEED, n_jobs=-1, verbose=-1
)
lgbm_sub.fit(
    tr_sub[FEAT_COLS], tr_sub[TARGET],
    eval_set=[(val_70[FEAT_COLS], val_70[TARGET])],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
m_lgbm_sub = compute_metrics(y_te, lgbm_sub.predict(te_eval[FEAT_COLS]))
t_lgbm_sub = round(time.time() - t0, 1)

# 2. Optimal Blend on validation split
weights = np.round(np.arange(0.0, 1.0001, 0.05), 2)
val_mae_curve = [
    compute_metrics(val_70[TARGET], w * tab_val_preds + (1 - w) * lgbm_extra_val)['MAE']
    for w in weights
]
w_best = float(weights[int(np.argmin(val_mae_curve))])
blend_te = w_best * tab_preds_median + (1 - w_best) * lgbm_extra_te
m_blend = compute_metrics(y_te, blend_te)
print(f'Optimal validation blend weight (TabPFN share): {w_best:.2f}')

def _fmt_row(name, rows, feats, m, t):
    return {
        'model': name,
        'train_rows': rows,
        'features': feats,
        'test_MAE': round(m['MAE'], 3),
        'test_RMSE': round(m['RMSE'], 3),
        'test_R2': round(m['R2'], 4),
        'runtime_s': t,
    }

parity_df = pd.DataFrame([
    _fmt_row('Meituan App Estimate', 0, 0, m_base, 0.0),
    _fmt_row(f'LightGBM ({TRAIN_SAMPLE_SIZE:,} rows)', TRAIN_SAMPLE_SIZE, len(FEAT_COLS), m_lgbm_sub, t_lgbm_sub),
    _fmt_row(f'TabPFN-v3.5 (1x{TRAIN_SAMPLE_SIZE:,}, mean)', TRAIN_SAMPLE_SIZE, len(FEAT_COLS), m_tab_mean, elapsed),
    _fmt_row(f'TabPFN-v3.5 (1x{TRAIN_SAMPLE_SIZE:,}, median)', TRAIN_SAMPLE_SIZE, len(FEAT_COLS), m_tab_median, elapsed),
    _fmt_row('LightGBM (377k, original features)', len(tr_70), len(FEATURES_COMPLETE), m_lgbm_orig, t_lgbm_orig),
    _fmt_row('LightGBM (377k, new features)', len(tr_70), len(FEAT_COLS), m_lgbm_extra, t_lgbm_extra),
    _fmt_row(f'Blend TabPFN x{w_best:.2f} + LightGBM', len(tr_70), len(FEAT_COLS), m_blend, round(elapsed + t_lgbm_extra, 1)),
])

parity_df.to_csv(CKPT_DIR / 'colab_v2_parity_comparison.csv', index=False)
display(parity_df)


## Stage 7: Learning Curve & Sample Size Scaling (Addressing "Gostaria de rodar mais")

To directly answer whether larger contexts improve TabPFN accuracy and how far we can scale on GPU, this stage runs a controlled learning curve across sample sizes: `[10_000, 25_000, 50_000, 75_000, 100_000]`.

Both TabPFN and LightGBM receive the exact same stratified training sample at each size, and predictions are evaluated on a fast stratified test slice (`LC_EVAL_SIZE = 10,000` rows).
The point `n = TRAIN_SAMPLE_SIZE` (100k) reuses the Stage 5 predictions (same stratified context: same seed and strata) instead of refitting the most expensive size.
Memory cleanup and exception frame isolation ensure seamless transitions between sizes without CUDA memory leaks.


In [ ]:
te_lc, lc_idx, _ = sample_dataframe(te_70, LC_EVAL_SIZE, stratify_col='_y_bin', seed=SEED)
# te_eval == te_70 (mesma ordem) quando TEST_EVAL_SIZE is None -> dá para reaproveitar o Stage 5
CAN_REUSE_MAIN = TEST_EVAL_SIZE is None
y_lc = te_lc[TARGET].to_numpy()
lc_rows = []
tabpfn_oom = False

print(f'Running Learning Curve on {len(te_lc):,} test rows across sizes: {LEARNING_CURVE_SIZES}...')

for n in LEARNING_CURVE_SIZES:
    if n > len(tr_70):
        break
    sub, _, _ = sample_dataframe(tr_70, n, stratify_col='_y_bin', seed=SEED)

    # LightGBM at size n
    t0 = time.time()
    lg = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.05, num_leaves=31, random_state=SEED, n_jobs=-1, verbose=-1
    ).fit(
        sub[FEAT_COLS], sub[TARGET],
        eval_set=[(val_70[FEAT_COLS], val_70[TARGET])],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    m_lg_n = compute_metrics(y_lc, lg.predict(te_lc[FEAT_COLS]))
    lc_rows.append({
        'n_train': n,
        'model': 'LightGBM',
        'MAE': round(m_lg_n['MAE'], 3),
        'RMSE': round(m_lg_n['RMSE'], 3),
        'R2': round(m_lg_n['R2'], 4),
        'runtime_s': round(time.time() - t0, 1),
    })

    # TabPFN at size n
    if tabpfn_oom:
        continue

    if n == TRAIN_SAMPLE_SIZE and CAN_REUSE_MAIN:
        m_tab_n = compute_metrics(y_lc, tab_preds_median[lc_idx])
        lc_rows.append({
            'n_train': n,
            'model': 'TabPFN-v3.5',
            'MAE': round(m_tab_n['MAE'], 3),
            'RMSE': round(m_tab_n['RMSE'], 3),
            'R2': round(m_tab_n['R2'], 4),
            'runtime_s': elapsed,
        })
        print(f'  n={n:>7,} | TabPFN MAE={m_tab_n["MAE"]:.3f} (reaproveitado do Stage 5) | LightGBM MAE={m_lg_n["MAE"]:.3f}')
        continue

    cfg_n = replace(BASE_CONFIG, sample_size=n, verbose=False)
    t0 = time.time()
    def _fit_predict_lc(c):
        mdl = TabPFNModel(c).fit(
            sub, sub[TARGET], num_cols=FEAT_COLS, stratify_col=None  # sub is already stratified
        )
        return mdl.predict(te_lc, output_type='median')

    try:
        p_tab = run_with_oom_retry(_fit_predict_lc, cfg_n)
        m_tab_n = compute_metrics(y_lc, p_tab)
        lc_rows.append({
            'n_train': n,
            'model': 'TabPFN-v3.5',
            'MAE': round(m_tab_n['MAE'], 3),
            'RMSE': round(m_tab_n['RMSE'], 3),
            'R2': round(m_tab_n['R2'], 4),
            'runtime_s': round(time.time() - t0, 1),
        })
        print(f'  n={n:>7,} | TabPFN MAE={m_tab_n["MAE"]:.3f} | LightGBM MAE={m_lg_n["MAE"]:.3f}')
    except torch.cuda.OutOfMemoryError:
        print(f'[OOM] TabPFN exceeded GPU memory at n={n:,}; skipping larger sizes.')
        tabpfn_oom = True
    finally:
        free_gpu()

lc_df = pd.DataFrame(lc_rows)
lc_df.to_csv(CKPT_DIR / 'colab_v2_learning_curve.csv', index=False)
display(lc_df.pivot(index='n_train', columns='model', values='MAE'))


## Stage 8: Diagnostic Visualizations & Artifact Packaging

This stage plots:
1. **Model Parity Comparison (MAE)**
2. **Context Scaling / Learning Curve (TabPFN vs LightGBM)**
3. **Predicted vs Actual Delivery Times**

All artifacts, JSON audit manifests, predictions Parquet, and figures are compressed into `tabpfn_colab_v2_artifacts.zip` and downloaded.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

# 1. Bar chart: Model Comparison (MAE)
df_plot = parity_df.sort_values('test_MAE', ascending=False).reset_index(drop=True)
colors = [
    '#94a3b8' if 'Baseline' in m or 'Estimate' in m
    else '#0284c7' if 'TabPFN' in m
    else '#16a34a' if 'LightGBM' in m
    else '#dc2626'
    for m in df_plot['model']
]
bars = axes[0].barh(df_plot['model'], df_plot['test_MAE'], color=colors, edgecolor='white')
axes[0].set_xlim(float(df_plot['test_MAE'].min()) - 0.25, min(float(df_plot['test_MAE'].max()) + 0.2, 10.0))
axes[0].set_title('Test MAE: Reference Split (70/30 Manhattan)', fontsize=12)
axes[0].set_xlabel('MAE (minutes - lower is better)')
for i, v in enumerate(df_plot['test_MAE']):
    axes[0].text(v + 0.03, i, f'{v:.3f}', va='center', fontsize=9)

# 2. Learning curve
if not lc_df.empty:
    for m_name, color, marker in [('TabPFN-v3.5', '#0284c7', 'o'), ('LightGBM', '#16a34a', 's')]:
        sub_m = lc_df[lc_df['model'] == m_name]
        if not sub_m.empty:
            axes[1].plot(sub_m['n_train'], sub_m['MAE'], label=m_name, color=color, marker=marker, lw=2, ms=6)
    axes[1].set_title('Learning Curve: MAE vs Training Context Size', fontsize=12)
    axes[1].set_xlabel('Training Context Size (N)')
    axes[1].set_ylabel('Test MAE (min)')
    axes[1].legend(frameon=True)

# 3. Predicted vs Actual
rng = np.random.default_rng(SEED)
vis_idx = rng.choice(len(y_te), size=min(3000, len(y_te)), replace=False)
axes[2].scatter(y_te[vis_idx], tab_preds_median[vis_idx], s=7, alpha=0.3, color='#0284c7', label='TabPFN (median)')
lim = [5, min(120, float(np.percentile(y_te, 99.5)))]
axes[2].plot(lim, lim, ls='--', color='#0f172a', lw=1.5, label='Ideal (Predicted = Actual)')
axes[2].set_xlim(lim)
axes[2].set_ylim(lim)
axes[2].set_title(f'Predicted vs Actual Delivery Time (n={TRAIN_SAMPLE_SIZE:,})', fontsize=12)
axes[2].set_xlabel('Actual delivery time (min)')
axes[2].set_ylabel('Predicted delivery time (min)')
axes[2].legend(frameon=True, loc='upper left')

plt.tight_layout()
fig_path = CKPT_DIR / 'colab_v2_tabpfn_diagnostics.png'
plt.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()

# Package artifacts
zip_path = CKPT_DIR / 'tabpfn_colab_v2_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(CKPT_DIR.glob('colab_v2_*')):
        zf.write(f, arcname=f.name)
print(f'Packaged Colab v2 artifacts to {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
